# 01 - Build Transfer Success Dataset

Output file:
- `data/processed/transfer_modeling_dataset.csv`

Row grain:
- 1 row = 1 player transfer


## 1) Imports


In [133]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd


## 2) Config

You can tune these values if needed.


In [134]:
ROOT = Path.cwd().resolve().parent
DATA_DIR = ROOT / 'data' / 'processed'
OUT_PATH = DATA_DIR / 'transfer_modeling_dataset.csv'

TOP5_LEAGUES = {'GB1', 'ES1', 'IT1', 'FR1', 'L1'}
FIXTURE_SEASONS = {2022, 2023} 

PRE_WINDOW_DAYS = 365
POST_WINDOW_DAYS = 1095  # about 3 seasons
MIN_POST_MINUTES_AVG = 900

print('ROOT:', ROOT)
print('Output:', OUT_PATH)


ROOT: D:\DataSince\project\ScoutRadar
Output: D:\DataSince\project\ScoutRadar\data\processed\transfer_modeling_dataset.csv


## 3) Load Base Tables


In [135]:
transfers = pd.read_csv(DATA_DIR / 'transfers_clean.csv')
players = pd.read_csv(DATA_DIR / 'players_clean.csv')
appearances = pd.read_csv(DATA_DIR / 'appearances_clean.csv')
valuations = pd.read_csv(DATA_DIR / 'player_valuations_clean.csv')
clubs = pd.read_csv(DATA_DIR / 'clubs_clean.csv')
competitions = pd.read_csv(DATA_DIR / 'competitions_clean.csv')

# print('transfers:', transfers.shape)
# print('players:', players.shape)
# print('appearances:', appearances.shape)
# print('valuations:', valuations.shape)
# print('clubs:', clubs.shape)
# print('competitions:', competitions.shape)


## 4) Basic Type Cleaning


In [136]:
transfers['transfer_date'] = pd.to_datetime(transfers['transfer_date'], errors='coerce')
# from "22/23" -> 2022 and 2023
s = transfers['transfer_season'].astype(str).str.extract(r'^\s*(\d{2})/(\d{2})\s*$')
transfers['season_start_year'] = (2000 + pd.to_numeric(s[0], errors='coerce')).astype('Int64')
transfers['season_end_year']   = (2000 + pd.to_numeric(s[1], errors='coerce')).astype('Int64')
transfers['transfer_fee_eur'] = pd.to_numeric(transfers['transfer_fee'], errors='coerce')
transfers['market_value_at_transfer_eur'] = pd.to_numeric(transfers['market_value_in_eur'], errors='coerce')

players['date_of_birth'] = pd.to_datetime(players['date_of_birth'], errors='coerce')
appearances['date'] = pd.to_datetime(appearances['date'], errors='coerce')
valuations['date'] = pd.to_datetime(valuations['date'], errors='coerce')
valuations['market_value_in_eur'] = pd.to_numeric(valuations['market_value_in_eur'], errors='coerce')

print('Null transfer_date:', transfers['transfer_date'].isna().sum())
print('Null transfer_season:', transfers['transfer_season'].isna().sum())
print('Null season_start_year:', transfers['season_start_year'].isna().sum())
print('Null date_birth:', players['date_of_birth'].isna().sum())
print('Null appearances_date:',appearances['date'].isna().sum())
print('Null valuations:',valuations['market_value_in_eur'].isna().sum())
transfers.head()

Null transfer_date: 0
Null transfer_season: 0
Null season_start_year: 0
Null date_birth: 0
Null appearances_date: 0
Null valuations: 0


,player_id,transfer_date,transfer_season,from_club_id,to_club_id,from_club_name,to_club_name,transfer_fee,market_value_in_eur,player_name,season_start_year,season_end_year,transfer_fee_eur,market_value_at_transfer_eur
0,467994,2030-06-30,25/26,5621,749,Reggiana,FC Empoli,0.0,700000.0,Luca Belardinelli,2025,2026,0.0,700000.0
1,784335,2027-07-18,27/28,6505,6502,Gimcheon Sangmu,Jeonbuk Hyundai,0.0,500000.0,Jun-soo Byeon,2027,2028,0.0,500000.0
2,402135,2027-07-04,27/28,6505,515,Gimcheon Sangmu,Without Club,0.0,350000.0,Jun-su Ahn,2027,2028,0.0,350000.0
3,716435,2027-07-04,27/28,6505,30925,Gimcheon Sangmu,Gwangju FC,0.0,325000.0,Kang-hyun Lee,2027,2028,0.0,325000.0
4,803933,2027-06-30,26/27,4467,14,TSV Hartberg,Austria Vienna,0.0,300000.0,Luca Pazourek,2026,2027,0.0,300000.0


## 5) Add Club/League Context and Filter to Scope

We focus on transfers into top-5 leagues and seasons 2022/2023.


In [137]:
club_leagues = clubs[['club_id', 'domestic_competition_id', 'average_age', 'squad_size']].copy()
club_leagues = club_leagues.rename(columns={
    'domestic_competition_id': 'league_code',
    'average_age': 'club_avg_age',
    'squad_size': 'club_squad_size',
})

base = transfers.copy()
base = base.merge(
    club_leagues[['club_id', 'league_code', 'club_avg_age', 'club_squad_size']].rename(columns={
        'club_id': 'from_club_id',
        'league_code': 'from_league_code',
        'club_avg_age': 'from_club_avg_age',
        'club_squad_size': 'from_club_squad_size',
    }),
    on='from_club_id', how='left'
)

base = base.merge(
    club_leagues[['club_id', 'league_code', 'club_avg_age', 'club_squad_size']].rename(columns={
        'club_id': 'to_club_id',
        'league_code': 'to_league_code',
        'club_avg_age': 'to_club_avg_age',
        'club_squad_size': 'to_club_squad_size',
    }),
    on='to_club_id', how='left'
)

base = base[base['to_league_code'].isin(TOP5_LEAGUES)].copy()
base = base[base['season_start_year'].isin(FIXTURE_SEASONS)].copy()
base = base[base['transfer_date'].notna()].copy()

base['transfer_row_id'] = np.arange(len(base), dtype=np.int64)
print('Scoped transfer rows:', len(base))
base[['player_id','transfer_date','transfer_season','from_league_code','to_league_code']].head()
base[base.isna().any(axis=1)]


Scoped transfer rows: 4995


,player_id,transfer_date,transfer_season,from_club_id,to_club_id,from_club_name,to_club_name,transfer_fee,market_value_in_eur,player_name,...,season_end_year,transfer_fee_eur,market_value_at_transfer_eur,from_league_code,from_club_avg_age,from_club_squad_size,to_league_code,to_club_avg_age,to_club_squad_size,transfer_row_id
29199,80444,2024-06-30,23/24,26091,405,Al-Duhail SC,Aston Villa,0.0,1225000.0,Philippe Coutinho,...,2024,0.0,1225000.0,NaN,NaN,NaN,GB1,28.1,25.0,8
29202,89086,2024-06-30,23/24,10586,430,Aris Limassol,Fiorentina,0.0,800000.0,Aleksandr Kokorin,...,2024,0.0,800000.0,NaN,NaN,NaN,IT1,26.1,27.0,10
29252,188436,2024-06-30,23/24,4031,8970,Cosenza,Frosinone,0.0,600000.0,Luigi Canotto,...,2024,0.0,600000.0,NaN,NaN,NaN,IT1,23.8,31.0,43
29254,190634,2024-06-30,23/24,5621,252,Reggiana,Genoa,0.0,350000.0,Marko Pajac,...,2024,0.0,350000.0,NaN,NaN,NaN,IT1,26.6,26.0,45
29265,200188,2024-06-30,23/24,332,252,Bari,Genoa,0.0,1225000.0,Mattia Aramu,...,2024,0.0,1225000.0,NaN,NaN,NaN,IT1,26.6,26.0,55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59126,988862,2022-07-01,22/23,11998,800,Atalanta U19,Atalanta BC,0.0,1000000.0,Moustapha Cissé,...,2023,0.0,1000000.0,NaN,NaN,NaN,IT1,27.3,24.0,4990
59227,1011200,2022-07-01,22/23,10872,40,G. Bordeaux B,G. Bordeaux,0.0,200000.0,Junior Mwanga,...,2023,0.0,200000.0,NaN,NaN,NaN,FR1,22.7,26.0,4991
60667,465677,2022-06-30,22/23,861,2921,Peñarol,Pescara,0.0,500000.0,Edgar Elizalde,...,2023,0.0,500000.0,NaN,NaN,NaN,IT1,24.6,28.0,4992
61390,653415,2022-06-05,22/23,10789,410,Braga B,Udinese,2000000.0,450000.0,Leonardo Buta,...,2023,2000000.0,450000.0,NaN,NaN,NaN,IT1,26.1,26.0,4993


## 6) Add Player Profile Features


In [138]:
player_cols = [
    'player_id', 'date_of_birth', 'position', 'sub_position', 'country_of_citizenship',
    'height_in_cm', 'foot'
]
base = base.merge(players[player_cols], on='player_id', how='left')

base['age_at_transfer'] = ((base['transfer_date'] - base['date_of_birth']).dt.days / 365.25).astype('float32')
base = base.rename(columns={
    'position': 'primary_position',
    'sub_position': 'secondary_position',
    'country_of_citizenship': 'nationality',
    'height_in_cm': 'height_cm',
})

base[['player_id','age_at_transfer','primary_position','secondary_position','nationality']].head()


,player_id,age_at_transfer,primary_position,secondary_position,nationality
0,328770,26.119097,Goalkeeper,Goalkeeper,Norway
1,12907,30.006845,Goalkeeper,Goalkeeper,Italy
2,33763,35.635864,Attack,Centre-Forward,Italy
3,41384,35.197811,Attack,Centre-Forward,Austria
4,42460,35.405888,Attack,Left Winger,Croatia


## 7) Pre-Transfer Performance Features (1-year window before transfer)


In [139]:
player_ids = set(base['player_id'].dropna().unique().tolist())
apps = appearances[appearances['player_id'].isin(player_ids)].copy()

apps_for_join = apps[['player_id','date','goals','assists','minutes_played','yellow_cards','red_cards']].copy()
apps_join = base[['transfer_row_id','player_id','transfer_date']].merge(apps_for_join, on='player_id', how='left')

mask_pre = (
    (apps_join['date'] < apps_join['transfer_date'])
    & (apps_join['date'] >= (apps_join['transfer_date'] - pd.to_timedelta(PRE_WINDOW_DAYS, unit='D')))
)
pre = apps_join[mask_pre].copy()

pre_agg = pre.groupby('transfer_row_id', as_index=False).agg(
    pre_minutes_total=('minutes_played', 'sum'),
    pre_goals_total=('goals', 'sum'),
    pre_assists_total=('assists', 'sum'),
    pre_yellow_cards_total=('yellow_cards', 'sum'),
    pre_red_cards_total=('red_cards', 'sum'),
)

base = base.merge(pre_agg, on='transfer_row_id', how='left')
print(base.shape)
for c in ['pre_minutes_total','pre_goals_total','pre_assists_total','pre_yellow_cards_total','pre_red_cards_total']:
    print (c,base[c].isna().sum())
    base[c] = base[c].fillna(0)

mins_safe = base['pre_minutes_total'].replace(0, np.nan)
base['pre_goals_per90'] = (90.0 * base['pre_goals_total'] / mins_safe).fillna(0).astype('float32')
base['pre_assists_per90'] = (90.0 * base['pre_assists_total'] / mins_safe).fillna(0).astype('float32')
base['pre_goal_contrib_per90'] = (base['pre_goals_per90'] + base['pre_assists_per90']).astype('float32')

base[['pre_minutes_total','pre_goals_total','pre_assists_total','pre_goals_per90','pre_assists_per90']].head()
print(base.isna().sum())


(4995, 33)
pre_minutes_total 1877
pre_goals_total 1877
pre_assists_total 1877
pre_yellow_cards_total 1877
pre_red_cards_total 1877
player_id                          0
transfer_date                      0
transfer_season                    0
from_club_id                       0
to_club_id                         0
from_club_name                     0
to_club_name                       0
transfer_fee                       0
market_value_in_eur                0
player_name                        0
season_start_year                  0
season_end_year                    0
transfer_fee_eur                   0
market_value_at_transfer_eur       0
from_league_code                1319
from_club_avg_age               1319
from_club_squad_size            1319
to_league_code                     0
to_club_avg_age                    0
to_club_squad_size                 0
transfer_row_id                    0
date_of_birth                      0
primary_position                   0
secondary_position

## 8) Post-Transfer Performance (for Target Construction)


In [140]:
apps_join_post = base[['transfer_row_id','player_id','transfer_date']].merge(apps_for_join, on='player_id', how='left')
post_end = apps_join_post['transfer_date'] + pd.to_timedelta(POST_WINDOW_DAYS, unit='D')

mask_post = (apps_join_post['date'] >= apps_join_post['transfer_date']) & (apps_join_post['date'] <= post_end)
post = apps_join_post[mask_post].copy()
post['year'] = post['date'].dt.year

post_agg = post.groupby('transfer_row_id', as_index=False).agg(
    post_minutes_total=('minutes_played', 'sum'),
    post_goals_total=('goals', 'sum'),
    post_assists_total=('assists', 'sum'),
    post_seasons_observed=('year', pd.Series.nunique),
)
post_agg['post_seasons_observed'] = post_agg['post_seasons_observed'].clip(lower=1)
post_agg['post_minutes_avg_per_season'] = post_agg['post_minutes_total'] / post_agg['post_seasons_observed']

post_mins_safe = post_agg['post_minutes_total'].replace(0, np.nan)
post_agg['post_goals_per90_avg'] = (90.0 * post_agg['post_goals_total'] / post_mins_safe).fillna(0)
post_agg['post_assists_per90_avg'] = (90.0 * post_agg['post_assists_total'] / post_mins_safe).fillna(0)

base = base.merge(
    post_agg[['transfer_row_id','post_minutes_avg_per_season','post_goals_per90_avg','post_assists_per90_avg']],
    on='transfer_row_id',
    how='left'
)
for c in ['post_minutes_avg_per_season','post_goals_per90_avg','post_assists_per90_avg']:
    base[c] = base[c].fillna(0).astype('float32')

base[['post_minutes_avg_per_season','post_goals_per90_avg','post_assists_per90_avg']].head()


,post_minutes_avg_per_season,post_goals_per90_avg,post_assists_per90_avg
0,263.500000,0.000000,0.000000
1,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000
3,545.333313,0.495110,0.165037
4,1757.666626,0.409634,0.409634


## 9) Market Value Criteria (Valuation at Transfer vs End of Window)


In [141]:
vals = valuations[valuations['player_id'].isin(player_ids)].copy()
vals = vals[['player_id','date','market_value_in_eur']].dropna(subset=['date'])

val_join = base[['transfer_row_id','player_id','transfer_date']].merge(vals, on='player_id', how='left')

# latest valuation on/before transfer date
v_before = val_join[val_join['date'] <= val_join['transfer_date']].copy()
v_before = v_before.sort_values(['transfer_row_id','date']).groupby('transfer_row_id', as_index=False).tail(1)
v_before = v_before[['transfer_row_id','market_value_in_eur']].rename(columns={'market_value_in_eur':'mv_before_transfer'})

# latest valuation within post window
val_join['window_end'] = val_join['transfer_date'] + pd.to_timedelta(POST_WINDOW_DAYS, unit='D')
v_after = val_join[(val_join['date'] >= val_join['transfer_date']) & (val_join['date'] <= val_join['window_end'])].copy()
v_after = v_after.sort_values(['transfer_row_id','date']).groupby('transfer_row_id', as_index=False).tail(1)
v_after = v_after[['transfer_row_id','market_value_in_eur']].rename(columns={'market_value_in_eur':'market_value_end_window_eur'})

base = base.merge(v_before, on='transfer_row_id', how='left')
base = base.merge(v_after, on='transfer_row_id', how='left')

# final market value at transfer: prefer transfer table value, else fallback to valuation snapshot
base['market_value_at_transfer_eur'] = base['market_value_at_transfer_eur'].fillna(base['mv_before_transfer'])


## 10) Build Criteria and Final Target


In [142]:
# 1) Minutes criterion
base['criteria_minutes'] = (base['post_minutes_avg_per_season'] >= MIN_POST_MINUTES_AVG).astype('int8')

# 2) Value criterion
base['criteria_value'] = (
    base['market_value_end_window_eur'].fillna(-1) >= base['market_value_at_transfer_eur'].fillna(np.inf)
).astype('int8')

# 3) Position KPI criterion
pos = base['primary_position'].fillna('Unknown').astype(str)
attack_mask = pos.eq('Attack')
mid_mask = pos.eq('Midfield')
def_mask = pos.eq('Defender')
gk_mask = pos.eq('Goalkeeper')

base['criteria_position_kpi'] = 0
base.loc[attack_mask, 'criteria_position_kpi'] = (
    (base.loc[attack_mask, 'post_goals_per90_avg'] >= 0.35)
    | (base.loc[attack_mask, 'post_assists_per90_avg'] >= 0.20)
).astype('int8')
base.loc[mid_mask, 'criteria_position_kpi'] = (
    base.loc[mid_mask, 'post_assists_per90_avg'] >= 0.18
).astype('int8')
base.loc[def_mask, 'criteria_position_kpi'] = (
    base.loc[def_mask, 'post_goals_per90_avg'] >= 0.08
).astype('int8')
base.loc[gk_mask, 'criteria_position_kpi'] = (
    base.loc[gk_mask, 'post_minutes_avg_per_season'] >= 1800
).astype('int8')

score = base['criteria_minutes'] + base['criteria_value'] + base['criteria_position_kpi']
base['transfer_success'] = (score >= 2).astype('int8')

base[['criteria_minutes','criteria_value','criteria_position_kpi','transfer_success']].head()


,criteria_minutes,criteria_value,criteria_position_kpi,transfer_success
0,0,1,0,0
1,0,0,0,0
2,0,0,0,0
3,0,1,1,1
4,1,1,1,1


## 11) Add Remaining Modeling Features and API Flags


In [143]:
league_strength = {'GB1': 5, 'ES1': 4, 'IT1': 3, 'FR1': 2, 'L1': 1}
base['from_is_top5'] = base['from_league_code'].isin(TOP5_LEAGUES).astype('int8')
base['to_is_top5'] = base['to_league_code'].isin(TOP5_LEAGUES).astype('int8')
base['league_tier_delta'] = (
    base['to_league_code'].map(league_strength).fillna(0)
    - base['from_league_code'].map(league_strength).fillna(0)
).astype('int8')

# API enrichment placeholders (until transfer-level API merge is added)
base['to_club_prev_season_rank'] = np.nan
base['to_club_current_season_ppg'] = np.nan
base['has_api_team_match'] = 0
base['has_api_prev_standing'] = 0
base['has_api_current_form'] = 0


## 12) Final Column Order


In [147]:
final_cols = [
    'transfer_row_id',
    'player_id',
    'transfer_date',
    'transfer_season',
    'from_club_id',
    'to_club_id',
    'from_club_name',
    'to_club_name',
    'transfer_fee_eur',
    'market_value_at_transfer_eur',
    'age_at_transfer',
    'primary_position',
    'secondary_position',
    'nationality',
    'height_cm',
    'foot',
    'pre_minutes_total',
    'pre_goals_total',
    'pre_assists_total',
    'pre_yellow_cards_total',
    'pre_red_cards_total',
    'pre_goals_per90',
    'pre_assists_per90',
    'pre_goal_contrib_per90',
    'from_league_code',
    'to_league_code',
    'from_is_top5',
    'to_is_top5',
    'league_tier_delta',
    'from_club_avg_age',
    'to_club_avg_age',
    'from_club_squad_size',
    'to_club_squad_size',
    'to_club_prev_season_rank',
    'to_club_current_season_ppg',
    'has_api_team_match',
    'has_api_prev_standing',
    'has_api_current_form',
    'post_minutes_avg_per_season',
    'post_goals_per90_avg',
    'post_assists_per90_avg',
    'market_value_end_window_eur',
    'criteria_minutes',
    'criteria_value',
    'criteria_position_kpi',
    'transfer_success',
]

final_df = base[final_cols].copy()

all_null_cols = [col for col in final_df.columns if final_df[col].isna().all()]
print('All-null columns:', all_null_cols)

final_df = final_df.drop(columns=all_null_cols)
final_df=final_df.dropna()
print('Final shape:', final_df.shape)
final_df.head()


All-null columns: ['to_club_prev_season_rank', 'to_club_current_season_ppg']
Final shape: (3581, 44)


,transfer_row_id,player_id,transfer_date,transfer_season,from_club_id,to_club_id,from_club_name,to_club_name,transfer_fee_eur,market_value_at_transfer_eur,...,has_api_prev_standing,has_api_current_form,post_minutes_avg_per_season,post_goals_per90_avg,post_assists_per90_avg,market_value_end_window_eur,criteria_minutes,criteria_value,criteria_position_kpi,transfer_success
0,0,328770,2024-07-25,23/24,501,415,Bodø/Glimt,Toulouse,0.0,450000.0,...,0,0,263.500000,0.000000,0.000000,500000.0,0,1,0,0
3,3,41384,2024-06-30,23/24,46,1025,Inter,Bologna,0.0,1225000.0,...,0,0,545.333313,0.495110,0.165037,2500000.0,0,1,1,1
4,4,42460,2024-06-30,23/24,447,148,Hajduk Split,Tottenham,0.0,1225000.0,...,0,0,1757.666626,0.409634,0.409634,1300000.0,1,1,1,1
5,5,60957,2024-06-30,23/24,105,33,Darmstadt 98,FC Schalke 04,0.0,500000.0,...,0,0,30.000000,0.000000,0.000000,300000.0,0,0,0,0
6,6,73794,2024-06-30,23/24,416,800,Torino,Atalanta,0.0,1225000.0,...,0,0,556.666687,0.377246,0.053892,3000000.0,0,1,1,1


## 13) Class Distribution Check


In [145]:
counts = final_df['transfer_success'].value_counts(dropna=False).sort_index()
ratios = final_df['transfer_success'].value_counts(normalize=True, dropna=False).sort_index()

print('Counts:')
print(counts)
print('Ratios:')
print(ratios)


Counts:
transfer_success
0    1606
1    1975
Name: count, dtype: int64
Ratios:
transfer_success
0    0.448478
1    0.551522
Name: proportion, dtype: float64


## 14) Save Dataset


In [146]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
final_df.to_csv(OUT_PATH, index=False)
print('Saved:', OUT_PATH)


Saved: D:\DataSince\project\ScoutRadar\data\processed\transfer_modeling_dataset.csv
